# E05 — F.cross_entropy

`F.cross_entropy` prend les **logits bruts** et les cibles, et fait en un
seul appel ce que faisaient `exp`, la normalisation, `log` et `mean`.

In [1]:
words = open('names.txt', 'r').read().splitlines()

In [2]:
chars = sorted(list(set(''.join(words))))
stoi = {s : i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

In [3]:
import torch
import torch.nn.functional as F

In [4]:
xs1, xs2, ys = [], [], []
for w in words:
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        xs1.append(stoi[ch1])
        xs2.append(stoi[ch2])
        ys.append(stoi[ch3])

xs = torch.tensor(xs1) * 27 + torch.tensor(xs2)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of exemples: ', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

number of exemples:  228146


## Même loss, mêmes gradients

In [5]:
gg = torch.Generator().manual_seed(2147483647)
Wn = torch.randn((729, 27), generator=gg, requires_grad=True)

logits = Wn[xs]
probs = logits.exp() / logits.exp().sum(1, keepdim=True)
loss_manuel = -probs[torch.arange(num), ys].log().mean()
loss_ce = F.cross_entropy(logits, ys)

Wn.grad = None
loss_manuel.backward(retain_graph=True)
g_manuel = Wn.grad.clone()

Wn.grad = None
loss_ce.backward()
g_ce = Wn.grad.clone()

d = (g_manuel - g_ce).abs()
echelle = g_manuel.abs().max()
cos = F.cosine_similarity(g_manuel.double().flatten(), g_ce.double().flatten(), dim=0)

print(f'loss manuelle           : {loss_manuel.item():.9f}')
print(f'loss cross_entropy      : {loss_ce.item():.9f}')
print(f'plus grand gradient en valeur abs: {echelle.item():.3e}')
print(f'plus grand ecart entre gradients : {d.max().item():.3e}')
print(f'ecart rapporte a cette echelle   : {(d.max() / echelle).item():.3e}')
print(f'cosinus entre les deux gradients : {cos.item():.10f}')
print(f'torch.allclose, tolerances defaut: {torch.allclose(g_manuel, g_ce)}')


loss manuelle           : 3.792776585
loss cross_entropy      : 3.792776346
plus grand gradient en valeur abs: 3.167e-02
plus grand ecart entre gradients : 5.802e-07
ecart rapporte a cette echelle   : 1.832e-05
cosinus entre les deux gradients : 0.9999999998
torch.allclose, tolerances defaut: False


In [6]:
%%timeit
logits = W[xs]
probs = logits.exp() / logits.exp().sum(1, keepdim=True)
loss = -probs[torch.arange(num), ys].log().mean()
W.grad = None
loss.backward()


15.9 ms ± 713 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%%timeit
loss = F.cross_entropy(W[xs], ys)
W.grad = None
loss.backward()

10.7 ms ± 254 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Stabilité numérique

In [8]:
def loss_manuelle(lg, y):
    p = lg.exp() / lg.exp().sum(1, keepdim=True)
    return -p[torch.arange(y.nelement()), y].log().mean().item()


y = torch.tensor([0])

for lg in [torch.tensor([[0.0, 1.0, 2.0]]),
           torch.tensor([[0.0, 100.0, 200.0]]),
           torch.tensor([[0.0, -100.0, -200.0]])]:
    print(lg.tolist(),
          '| manuel', round(loss_manuelle(lg, y), 4),
          '| cross_entropy', round(F.cross_entropy(lg, y).item(), 4))

[[0.0, 1.0, 2.0]] | manuel 2.4076 | cross_entropy 2.4076
[[0.0, 100.0, 200.0]] | manuel inf | cross_entropy 200.0
[[0.0, -100.0, -200.0]] | manuel -0.0 | cross_entropy 0.0


In [9]:
# la loss ne dépend que des écarts entre logits : ajouter une constante ne change rien
lg = torch.tensor([[0.0, 1.0, 2.0]])
for c in [0.0, 50.0, 100.0]:
    print(c, '| manuel', loss_manuelle(lg + c, y), '| cross_entropy', F.cross_entropy(lg + c, y).item())

0.0 | manuel 2.4076058864593506 | cross_entropy 2.4076058864593506
50.0 | manuel 2.4076058864593506 | cross_entropy 2.4076058864593506
100.0 | manuel nan | cross_entropy 2.4076058864593506


## Entraînement

In [10]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

losses = []

for k in range(2001):
    loss = F.cross_entropy(W[xs], ys)
    losses.append(loss.item())
    if k % 100 == 0:
        print(k, loss.item())

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

0 3.792776346206665
100 2.5085482597351074
200 2.3814942836761475
300 2.3280222415924072
400 2.297999620437622
500 2.278773307800293
600 2.265404462814331
700 2.255554676055908
800 2.247979164123535
900 2.241960287094116
1000 2.237057685852051
1100 2.2329838275909424
1200 2.229543685913086
1300 2.2265994548797607
1400 2.224050521850586
1500 2.2218210697174072
1600 2.2198538780212402
1700 2.218104362487793
1800 2.2165369987487793
1900 2.2151246070861816
2000 2.2138442993164062


In [11]:
g = torch.Generator().manual_seed(2147483647)

for k in range(20):
    out = []
    ix1, ix2 = 0, 0
    while True:
        ctx = ix1 * 27 + ix2
        p = F.softmax(W[[ctx]], dim=1)
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1, ix2 = ix2, ix3
    print(''.join(out))

ce.
bra.
jalius.
rochityharlonimittain.
luwan.
ka.
da.
samiyah.
javer.
gotai.
moriellavorie.
teda.
kaley.
maside.
en.
aviony.
fobspihaniven.
tahlas.
kashrxdleenlen.
al.
